# Task 8 - TF-IDF + SVM

Baseline cổ điển: biến văn bản thành vector TF-IDF rồi phân loại bằng SVM tuyến tính (LinearSVC). Minh hoạ cho cả 3 subtask của Task 8 (StereoQueerEval).

In [ ]:
!pip install -q scikit-learn pandas

In [ ]:
!git clone -q https://github.com/hoanhh4977/SemEval2027.git
%cd SemEval2027/

In [ ]:
import pandas as pd

data = pd.read_csv("data/task8/train/StereoQueerEval_EN_training.tsv", sep="\t")
data.head()

## Chuẩn bị nhãn cho 3 subtask

In [ ]:
# Subtask A - Stereotype Detection (nhị phân)
label2id_A = {"yes": 0, "no": 1}
id2label_A = {v: k for k, v in label2id_A.items()}
data["label_A"] = data["stereotype"].map(label2id_A)

# Subtask B - Hate Speech Classification (3 lớp)
label2id_B = {"no": 0, "yes_implicit": 1, "yes_explicit": 2}
id2label_B = {v: k for k, v in label2id_B.items()}
data["label_B"] = data["hate_speech"].map(label2id_B)

# Subtask C - Target & Identity Identification (multilabel, chỉ trên comment hateful)
def parse_target(x):
    if x == "none":
        return []
    scope, identities = x.split("_", 1)
    return [scope] + identities.split(",")

data["target_labels"] = data["target"].apply(parse_target)
data_c = data[data["hate_speech"] != "no"].reset_index(drop=True)
print(f"Subtask C chỉ dùng {len(data_c)}/{len(data)} comment hateful")

## Chia Train/Test

In [ ]:
from sklearn.model_selection import train_test_split

train_A, test_A = train_test_split(data, test_size=0.2, random_state=42, stratify=data["label_A"])
train_B, test_B = train_test_split(data, test_size=0.2, random_state=42, stratify=data["label_B"])
train_C, test_C = train_test_split(data_c, test_size=0.2, random_state=42)

print(f"A: {len(train_A)}/{len(test_A)}  B: {len(train_B)}/{len(test_B)}  C: {len(train_C)}/{len(test_C)}")

## Hàm dùng chung cho cả 3 subtask

Chỉ cần đổi vectorizer và cờ `multilabel` là dùng lại được cho mọi subtask — minh hoạ cách viết code tái sử dụng thay vì copy-paste 3 lần.

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import LinearSVC
from sklearn.multiclass import OneVsRestClassifier
from sklearn.preprocessing import MultiLabelBinarizer
from sklearn.metrics import classification_report, f1_score, hamming_loss

def run_svm_subtask(train_df, test_df, label_col, vectorizer, target_names=None, multilabel=False):
    X_train = vectorizer.fit_transform(train_df["yt_comment"])
    X_test = vectorizer.transform(test_df["yt_comment"])

    if multilabel:
        mlb = MultiLabelBinarizer()
        y_train = mlb.fit_transform(train_df[label_col])
        y_test = mlb.transform(test_df[label_col])
        print("Các nhãn:", list(mlb.classes_))

        clf = OneVsRestClassifier(LinearSVC(class_weight="balanced"))
        clf.fit(X_train, y_train)
        pred = clf.predict(X_test)
        print(f"F1-micro: {f1_score(y_test, pred, average='micro'):.3f}  "
              f"F1-macro: {f1_score(y_test, pred, average='macro'):.3f}  "
              f"Hamming loss: {hamming_loss(y_test, pred):.3f}")
        return clf, mlb

    clf = LinearSVC(class_weight="balanced")
    clf.fit(X_train, train_df[label_col])
    pred = clf.predict(X_test)
    print(classification_report(test_df[label_col], pred, target_names=target_names))
    return clf, vectorizer

## Subtask A: Stereotype Detection

In [ ]:
run_svm_subtask(
    train_A, test_A, "label_A",
    TfidfVectorizer(ngram_range=(1, 2), min_df=2, max_features=20000),
    target_names=[id2label_A[0], id2label_A[1]],
)

## Subtask B: Hate Speech Classification

In [ ]:
run_svm_subtask(
    train_B, test_B, "label_B",
    TfidfVectorizer(ngram_range=(1, 2), min_df=2, max_features=20000),
    target_names=[id2label_B[i] for i in sorted(id2label_B)],
)

## Subtask C: Target & Identity Identification (multilabel)

Mỗi comment hateful có thể mang nhiều nhãn cùng lúc (ví dụ vừa "group" vừa "l", "g"), nên đây là bài toán multilabel. `MultiLabelBinarizer` biến danh sách nhãn thành vector nhị phân nhiều chiều, còn `OneVsRestClassifier` huấn luyện một bộ phân loại nhị phân riêng cho từng nhãn.

In [ ]:
run_svm_subtask(
    train_C, test_C, "target_labels",
    TfidfVectorizer(ngram_range=(1, 2), min_df=2, max_features=20000),
    multilabel=True,
)